### Case Study 1: Loop Avoidance

The optimal solution should be to select enode 2 and 3 (in the figure below) with total cost 3.0 and delay 2.0.

It avoids the loop formed by enode 1 and 4 even though they have lower cost in total.

![example1](example1.jpg)

In [9]:
import gurobipy as grb

# Target Maximum Delay
D = 3.0
# Big M
M = 1000.0

model = grb.Model("extraction")
model.setParam("OutputFlag", 0)

# Costs
cost_x = [1.0, 1.0, 2.0, 1.0]

# Delays
delay_x = [4.0, 1.0, 1.0, 1.0]

# Variables
x = model.addVars(4, vtype=grb.GRB.BINARY, name="x")    # enode decision
z = model.addVars(2, vtype=grb.GRB.BINARY, name="z")    # eclass decision

delta = model.addVars(2, vtype=grb.GRB.CONTINUOUS, name="delta")    # delay of each eclass

def get_bin_con_prod(model: grb.Model, bin_var: grb.Var, con_var: grb.Var, name: str) -> grb.Var:
    """ Add the product of a binary variable and a continuous variable to the model.
        Returns the product variable.
    """
    prod_var = model.addVar(lb=0, vtype=grb.GRB.CONTINUOUS, name=f"{name}_prod")
    model.addConstr(prod_var <= M * bin_var, name=f"{name}_c1")
    model.addConstr(prod_var <= con_var, name=f"{name}_c2")
    model.addConstr(prod_var >= con_var - M * (1 - bin_var), name=f"{name}_c3")
    return prod_var

# Constraints
# Root Constraint
model.addConstr(z[0] >= 1, name="root")

# Child Constraints
model.addConstr(x[1] <= z[1], name="child_0")

# Eclass Constraints
model.addConstr(z[0] <= x[0] + x[1], name="eclass_0")
model.addConstr(z[1] <= x[2] + x[3], name="eclass_1")

# Eclass Delay Constraints
model.addConstr(delta[0] - x[0] * delay_x[0] >= 0, name="eclass_delay_0")
model.addConstr(delta[0] - x[1] * delay_x[1] >= get_bin_con_prod(model, x[1], delta[1], "tmp_0"), name="eclass_delay_1")
model.addConstr(delta[1] - x[2] * delay_x[2] >= 0, name="eclass_delay_2")
model.addConstr(delta[1] - x[3] * delay_x[3] >= get_bin_con_prod(model, x[3], delta[0], "tmp_1"), name="eclass_delay_3")

# Output Delay Constraint
model.addConstr(delta[0] <= D, name="output_delay")

# Objective: Minimize cost
model.setObjective(grb.quicksum(cost_x[i] * x[i] for i in range(4)), grb.GRB.MINIMIZE)

model.optimize()

# Print solution
if model.status == grb.GRB.OPTIMAL:
    print("\nOptimal solution found:\n")
    for v in model.getVars():
        if v.X > 0.5:
            print(f"{v.VarName}: {v.X}")
    print(f"\nOptimal objective value: {model.objVal}")


Optimal solution found:

x[1]: 1.0
x[2]: 1.0
z[0]: 1.0
z[1]: 1.0
delta[0]: 3.0
delta[1]: 1.0
tmp_0_prod: 1.0

Optimal objective value: 3.0


### Case Study 2: Register Selection

In this case, the optimal solution selects register 2 (in the figure below) to meet the delay constraint.

![example2](example2.jpg)

In [10]:
import gurobipy as grb

# Target Maximum Delay
D = 3.0
# Big M
M = 1000.0

model = grb.Model("extraction")
model.setParam("OutputFlag", 0)

# Costs
cost_x = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
cost_y = [2.0, 2.0]

# Delays
# delay_x = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
delay_x = [1.0, 1.0, 3.0, 1.0, 1.0, 1.0]

# Variables
x = model.addVars(6, vtype=grb.GRB.BINARY, name="x")    # enode decision
y = model.addVars(2, vtype=grb.GRB.BINARY, name="y")    # ff decision
z = model.addVars(4, vtype=grb.GRB.BINARY, name="z")    # eclass decision

delta = model.addVars(4, vtype=grb.GRB.CONTINUOUS, name="delta")    # delay of each eclass

def get_bin_con_prod(model: grb.Model, bin_var: grb.Var, con_var: grb.Var, name: str) -> grb.Var:
    """ Add the product of a binary variable and a continuous variable to the model.
        Returns the product variable.
    """
    prod_var = model.addVar(lb=0, vtype=grb.GRB.CONTINUOUS, name=f"{name}_prod")
    model.addConstr(prod_var <= M * bin_var, name=f"{name}_c1")
    model.addConstr(prod_var <= con_var, name=f"{name}_c2")
    model.addConstr(prod_var >= con_var - M * (1 - bin_var), name=f"{name}_c3")
    return prod_var

# Constraints
# Root Constraint
model.addConstr(z[1] >= 1, name="root")

# Child Constraints
model.addConstr(x[0] <= z[2], name="child_0")
model.addConstr(x[1] <= z[2], name="child_1")
model.addConstr(x[1] <= z[3], name="child_2")
model.addConstr(x[2] <= z[3], name="child_3")

# Eclass Constraints
model.addConstr(z[0] <= x[0], name="eclass_0")
model.addConstr(z[1] <= x[1] + x[2], name="eclass_1")
model.addConstr(z[2] <= x[3] + x[4], name="eclass_2")
model.addConstr(z[3] <= x[5], name="eclass_3")

# FF Constraints
model.addConstr(y[0] <= z[0], name="ff_d_0")
model.addConstr(y[1] <= z[1], name="ff_d_1")
model.addConstr(x[5] <= y[1], name="ff_q_0")
model.addConstr(x[4] <= y[1], name="ff_q_1")
model.addConstr(x[3] <= y[0], name="ff_q_2")

# FF Delay Constraints
model.addConstr(get_bin_con_prod(model, y[0], delta[0], "tmp_0") <= D, name="ff_delay_0")
model.addConstr(get_bin_con_prod(model, y[1], delta[1], "tmp_1") <= D, name="ff_delay_1")

# Output Delay Constraint
model.addConstr(delta[1] <= D, name="output_delay")

# Eclass Delay Constraints
model.addConstr(delta[0] - x[0] * delay_x[0] >= get_bin_con_prod(model, x[0], delta[2], "tmp_2"), name="eclass_delay_0")
model.addConstr(delta[1] - x[1] * delay_x[1] >= get_bin_con_prod(model, x[1], delta[2], "tmp_3"), name="eclass_delay_1")
model.addConstr(delta[1] - x[1] * delay_x[1] >= get_bin_con_prod(model, x[1], delta[3], "tmp_4"), name="eclass_delay_2")
model.addConstr(delta[1] - x[2] * delay_x[2] >= get_bin_con_prod(model, x[2], delta[3], "tmp_5"), name="eclass_delay_3")
model.addConstr(delta[2] >= x[3] * delay_x[3], name="eclass_delay_4")
model.addConstr(delta[2] >= x[4] * delay_x[4], name="eclass_delay_5")
model.addConstr(delta[3] >= x[5] * delay_x[5], name="eclass_delay_6")

# Objective: Minimize cost
model.setObjective(grb.quicksum(cost_x[i] * x[i] for i in range(6)) + grb.quicksum(cost_y[j] * y[j] for j in range(2)), grb.GRB.MINIMIZE)

model.optimize()

# Print solution
if model.status == grb.GRB.OPTIMAL:
    print("\nOptimal solution found:\n")
    for v in model.getVars():
        if v.X > 0.5:
            print(f"{v.VarName}: {v.X}")
    print(f"\nOptimal objective value: {model.objVal}")


Optimal solution found:

x[1]: 1.0
x[4]: 1.0
x[5]: 1.0
y[1]: 1.0
z[1]: 1.0
z[2]: 1.0
z[3]: 1.0
delta[0]: 1000.0
delta[1]: 3.0
delta[2]: 1.0
delta[3]: 1.0
tmp_1_prod: 3.0
tmp_3_prod: 1.0
tmp_4_prod: 1.0

Optimal objective value: 5.0
